# External validation and dataset shift: worked solution

Worked solution for the exercise in the lesson **[External validation and dataset shift](https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift)** (*Evaluation, safety & deployment* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](01-external-validation-dataset-shift.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** External validation tests a model on data from a different hospital, population or period than the one it was built on. Dataset shift, such as different patients, a different lab assay or a different baseline risk, often leaves a model's ranking ability (AUROC) roughly intact while making its predicted risks wrong. Plan external validation with enough outcome events, report calibration as well as discrimination, and be ready to recalibrate or update the model for each new setting.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Simulating four settings

Develop at one synthetic site; test under three kinds of shift.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

predictors = ["age", "egfr", "hba1c", "albuminuria"]

def site(n, seed, age_mean=58, assay_bias=0.0, baseline=-2.0):
    """Synthetic cohort: 3-year risk of a kidney outcome. assay_bias shifts measured eGFR only."""
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"age": rng.normal(age_mean, 10, n).round(),
                      "true_egfr": rng.normal(70, 20, n).clip(15, 120),
                      "hba1c": rng.normal(7.8, 1.3, n).round(1),
                      "albuminuria": rng.binomial(1, 0.3, n)})
    logit = (baseline + 0.02 * (d.age - 58) - 0.04 * (d.true_egfr - 70)
             + 0.3 * (d.hba1c - 7.8) + 1.0 * d.albuminuria)
    d["outcome"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    d["egfr"] = (d.true_egfr + assay_bias).round()          # what the lab reports
    return d

dev = site(6000, seed=1)
model = LogisticRegression(max_iter=1000).fit(dev[predictors], dev.outcome)

def evaluate(name, d):
    p = model.predict_proba(d[predictors])[:, 1]
    recal = LogisticRegression(penalty=None).fit(np.log(p / (1 - p)).reshape(-1, 1), d.outcome)
    print(f"{name:<28} n={len(d):>5}  outcome {d.outcome.mean():5.1%}  "
          f"AUROC {roc_auc_score(d.outcome, p):.3f}  predicted {p.mean():5.1%}  "
          f"slope {recal.coef_[0][0]:.2f}")

evaluate("Internal (same hospital)", site(3000, seed=2))
evaluate("Older population", site(3000, seed=3, age_mean=70))
evaluate("Different eGFR assay (+15)", site(3000, seed=4, assay_bias=15))
evaluate("Lower baseline risk", site(3000, seed=5, baseline=-2.8))

# Internal (same hospital)     n= 3000  outcome 18.7%  AUROC 0.734  predicted 18.8%  slope 0.90
# Older population             n= 3000  outcome 23.0%  AUROC 0.757  predicted 23.5%  slope 1.02
# Different eGFR assay (+15)   n= 3000  outcome 18.4%  AUROC 0.759  predicted 12.0%  slope 1.03
# Lower baseline risk          n= 3000  outcome 11.3%  AUROC 0.759  predicted 19.4%  slope 0.99

### Small validation samples are uncertain

Continuing from the previous block: bootstrap confidence interval for AUROC in a 400-patient sample.

In [ ]:
small = site(400, seed=6)
p = model.predict_proba(small[predictors])[:, 1]
rng = np.random.default_rng(0)
boot = []
for _ in range(1000):
    i = rng.integers(0, len(small), len(small))
    if small.outcome.iloc[i].nunique() == 2:
        boot.append(roc_auc_score(small.outcome.iloc[i], p[i]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"External sample of {len(small)} patients, {small.outcome.sum()} events: "
      f"AUROC {roc_auc_score(small.outcome, p):.3f} (95% CI {lo:.3f} to {hi:.3f})")

# External sample of 400 patients, 84 events: AUROC 0.750 (95% CI 0.690 to 0.807)

## Exercise

**Goal:** Explore how different kinds of dataset shift affect a clinical prediction model, and fix what can be fixed.

**Data:** Synthetic sites generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the four-site comparison and draw calibration plots for each site.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

sites = {
    "Internal (same hospital)": site(3000, seed=2),
    "Older population": site(3000, seed=3, age_mean=70),
    "Different eGFR assay (+15)": site(3000, seed=4, assay_bias=15),
    "Lower baseline risk": site(3000, seed=5, baseline=-2.8),
}
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, d in sites.items():
    evaluate(name, d)
    obs, pred = calibration_curve(d.outcome, model.predict_proba(d[predictors])[:, 1], n_bins=10, strategy="quantile")
    ax.plot(pred, obs, "o-", label=name)
ax.plot([0, 0.7], [0, 0.7], ":", color="grey")
ax.set(xlabel="Predicted risk", ylabel="Observed risk", title="Calibration by site")
ax.legend(fontsize=8)
plt.show()

AUROC barely changes between sites, but calibration does: at the assay-shift site the model under-predicts
(a higher reported eGFR looks reassuring), and at the low-risk site it over-predicts. A discrimination-only
validation would have passed both.

### Step 2

Add a concept-shift site where albuminuria matters half as much, and check AUROC and calibration.

In [ ]:
def concept_site(n, seed, albuminuria_effect=0.5):
    """Same predictors, but albuminuria matters half as much for the outcome."""
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"age": rng.normal(58, 10, n).round(), "true_egfr": rng.normal(70, 20, n).clip(15, 120),
                      "hba1c": rng.normal(7.8, 1.3, n).round(1), "albuminuria": rng.binomial(1, 0.3, n)})
    logit = (-2.0 + 0.02 * (d.age - 58) - 0.04 * (d.true_egfr - 70) + 0.3 * (d.hba1c - 7.8)
             + albuminuria_effect * d.albuminuria)
    d["outcome"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    d["egfr"] = d.true_egfr.round()
    return d

concept = concept_site(3000, seed=7)
evaluate("Concept shift (albuminuria x0.5)", concept)
p = model.predict_proba(concept[predictors])[:, 1]
for a in [0, 1]:
    m = concept.albuminuria == a
    print(f"  albuminuria={a}: predicted {p[m].mean():.1%}, observed {concept.outcome[m].mean():.1%}")

The overall numbers look only mildly worse, but the subgroup check shows the problem: patients with albuminuria
are now over-predicted. Concept shift changes the relationship between a predictor and the outcome, so
recalibrating the whole model cannot fix it; the coefficient itself has to be updated, or the model refitted.

### Step 3

Recalibrate the model at the assay-shift and low-risk sites using 500 local patients, and evaluate on the rest.

In [ ]:
from scipy.optimize import brentq

def recalibrated_predictions(local, rest, slope_too=True):
    lo_local = np.log((pl := model.predict_proba(local[predictors])[:, 1]) / (1 - pl))
    lo_rest = np.log((pr := model.predict_proba(rest[predictors])[:, 1]) / (1 - pr))
    if slope_too:
        r = LogisticRegression(penalty=None).fit(lo_local.reshape(-1, 1), local.outcome)
        return r.predict_proba(lo_rest.reshape(-1, 1))[:, 1]
    a = brentq(lambda a: np.sum(local.outcome - 1 / (1 + np.exp(-(lo_local + a)))), -10, 10)
    return 1 / (1 + np.exp(-(lo_rest + a)))

for name in ["Different eGFR assay (+15)", "Lower baseline risk"]:
    d = sites[name]
    local, rest = d.iloc[:500], d.iloc[500:]
    before = model.predict_proba(rest[predictors])[:, 1]
    after = recalibrated_predictions(local, rest)
    print(f"{name:<28} observed {rest.outcome.mean():.1%}; predicted before {before.mean():.1%}, after {after.mean():.1%}; "
          f"AUROC {roc_auc_score(rest.outcome, before):.3f} -> {roc_auc_score(rest.outcome, after):.3f}")

Recalibration on 500 local patients corrects the assay shift almost exactly (18.8% predicted against 18.4%
observed) and brings the low-risk site much closer (19.5% to 13.3%, against 10.9%); with only about 55 local
events, the correction is itself imprecise. AUROC does not change, because the ranking was never the problem. For the
assay shift there is also a better fix: correct the eGFR values (or the assay's offset) before they reach the
model, so the input means what it meant at development.

### Step 4

Repeat the bootstrap for samples of 200, 400 and 2,000 patients and plot how the interval narrows.

In [ ]:
sizes = [200, 400, 2000]
intervals = []
rng = np.random.default_rng(0)
for n_ in sizes:
    sample = site(n_, seed=100 + n_)
    pp = model.predict_proba(sample[predictors])[:, 1]
    boot = []
    for _ in range(1000):
        i = rng.integers(0, n_, n_)
        if sample.outcome.iloc[i].nunique() == 2:
            boot.append(roc_auc_score(sample.outcome.iloc[i], pp[i]))
    lo, hi = np.percentile(boot, [2.5, 97.5])
    intervals.append((n_, sample.outcome.sum(), roc_auc_score(sample.outcome, pp), lo, hi))
    print(f"n={n_:>5}  events {sample.outcome.sum():>4}  AUROC {intervals[-1][2]:.3f}  95% CI {lo:.3f} to {hi:.3f}  (width {hi - lo:.3f})")

fig, ax = plt.subplots(figsize=(5, 3.5))
for n_, ev, auc, lo, hi in intervals:
    ax.errorbar(n_, auc, yerr=[[auc - lo], [hi - auc]], fmt="o", capsize=4)
ax.set(xscale="log", xlabel="Validation sample size (patients)", ylabel="AUROC with 95% CI", xticks=sizes)
ax.set_xticklabels([str(s) for s in sizes])
plt.show()

The interval narrows roughly with the square root of the sample size, and what really matters is the number of
events. Older rules of thumb asked for at least 100 to 200 events in a validation study; formal sample-size
methods for external validation (Riley and colleagues, 2021) often require more, especially to estimate
calibration precisely. Many published validations are smaller.

### Step 5

Write the validation plan you would require before deploying this model at a new hospital.

Example validation plan before deploying this kidney-risk model at a new hospital:

1. **Intended use and population** written down: adults with type 2 diabetes in outpatient care, risk used to
   prioritise nephrology review, not to withhold care.
2. **Local data check**: same predictor definitions, units and assays (eGFR equation and laboratory method), and
   the same outcome definition and time horizon.
3. **Retrospective local validation** on recent patients, with enough events (aim for at least 100–200, more for
   calibration): AUROC, calibration-in-the-large, calibration slope and a calibration plot, decision curve at the
   intended threshold, all with confidence intervals.
4. **Subgroups**: age bands, sex, ethnicity or other locally relevant groups, albuminuria status, and eGFR bands.
5. **Pre-specified acceptance criteria** (for example calibration slope 0.8–1.2 and observed/expected 0.8–1.25);
   if calibration fails but discrimination holds, recalibrate on local data and re-validate on separate patients.
6. **Silent period**: run the model alongside care without showing results for a few months, then compare.
7. **Monitoring plan** in place before go-live (see the monitoring lesson), with an owner and a switch-off rule.

---

Lesson: [https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift](https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.